
# 📘 NLP Pipeline – Arabic PDF Processing

This notebook:
- Extracts text from a PDF
- Cleans Arabic text
- Tokenizes and removes stopwords
- Converts text to TF-IDF

In [40]:
#  1: Installation des prérequis
!pip install PyPDF2 nltk scikit-learn pandas




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
#  2: Import des bibliothèques
import PyPDF2
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Téléchargement des stopwords arabes NLTK (si disponible)
try:
    nltk.download('stopwords')
    arabic_stopwords_nltk = set(stopwords.words('arabic'))
except:
    arabic_stopwords_nltk = set()



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [42]:
#  3: Stopwords arabes complets (fallback manuel)
ARABIC_STOPWORDS = arabic_stopwords_nltk or {
    'من', 'الى', 'إلى', 'عن', 'على', 'في', 'هذا', 'هذه', 'ذلك', 'تلك', 'كان', 'كانت',
    'و', 'ف', 'ثم', 'ل', 'ب', 'ك', 'ما', 'لا', 'لم', 'لن', 'إن', 'إذا', 'عند',
    'بين', 'مع', 'حتى', 'خلال', 'دون', 'بعد', 'قبل', 'أثناء', 'هو', 'هي', 'هم',
    'هن', 'نحن', 'أنت', 'أنتم', 'أنا', 'أن', 'التي', 'الذي', 'الذين', 'أو', 'أي',
    'كل', 'بعض', 'هذه', 'به', 'لها', 'لهم', 'عليها', 'عليهم', 'كما', 'وقد', 'قد',
    'أو', 'ولقد', 'فقد', 'فإن', 'فهو', 'فهي', 'وكان', 'وكانت', 'الا', 'إلا', 'ولا',
    'وما', 'وأن', 'وبين', 'ففي', 'فمن', 'فكان', 'لكن', 'علي', 'إلي', 'لدى', 'ثمة',
    'حين', 'حيث', 'بما', 'بأن', 'لأن', 'لما', 'أين', 'كيف', 'متى', 'لماذا'
}



In [43]:
#  4: Fonction générique d'extraction PDF
def extract_text_from_pdf(pdf_path):
    """
    Extrait tout le texte d'un fichier PDF
    Args:
        pdf_path (str): Chemin vers le fichier PDF
    Returns:
        str: Texte extrait
    """
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page_num, page in enumerate(reader.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Attention: Page {page_num} - texte non extractible")
        print(f"Extraction réussie: {len(reader.pages)} pages, {len(text)} caractères")
    except Exception as e:
        print(f"Erreur extraction PDF: {e}")
    return text



In [44]:
#  5: Fonction générique de nettoyage texte arabe
def clean_arabic_text(text):
    """
    Nettoie le texte arabe
    Args:
        text (str): Texte brut
    Returns:
        str: Texte nettoyé
    """
    # Suppression des chiffres
    text = re.sub(r'[0-9]', '', text)
    
    # Suppression des caractères non arabes (garde lettres arabes, espace, point)
    # Plage Unicode arabe: \u0600-\u06FF
    text = re.sub(r'[^\u0600-\u06FF\s\.\,\!\\?]', ' ', text)
    
    # Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [45]:

#  6: Tokenisation et filtrage
def tokenize_and_filter(text, stopwords_set, min_word_length=2):
    """
    Tokenise et filtre le texte arabe
    Args:
        text (str): Texte nettoyé
        stopwords_set (set): Ensemble des stopwords
        min_word_length (int): Longueur minimale des mots
    Returns:
        list: Liste des tokens filtrés
    """
    # Tokenisation par espaces
    tokens = text.split()
    
    # Filtrage
    filtered_tokens = []
    for token in tokens:
        if len(token) >= min_word_length and token not in stopwords_set:
            filtered_tokens.append(token)
    
    return filtered_tokens


In [46]:

#  7: Pipeline NLP arabe générique
class ArabicNLPPipeline:
    """
    Pipeline NLP complet pour textes arabes
    """
    
    def __init__(self, pdf_path=None, custom_text=None):
        self.pdf_path = pdf_path
        self.custom_text = custom_text
        self.raw_text = ""
        self.cleaned_text = ""
        self.tokens = []
        self.stopwords = ARABIC_STOPWORDS
        self.tfidf_matrix = None
        self.vectorizer = None
        
    def load_data(self):
        """Charge les données depuis PDF ou texte direct"""
        if self.pdf_path:
            self.raw_text = extract_text_from_pdf(self.pdf_path)
        elif self.custom_text:
            self.raw_text = self.custom_text
        else:
            raise ValueError("Fournir pdf_path ou custom_text")
        return self
    
    def preprocess(self, min_word_length=2):
        """Prétraite le texte: nettoyage + tokenisation"""
        # Nettoyage
        self.cleaned_text = clean_arabic_text(self.raw_text)
        # Tokenisation et filtrage
        self.tokens = tokenize_and_filter(self.cleaned_text, self.stopwords, min_word_length)
        return self
    
    def vectorize(self, additional_docs=None):
        """Vectorisation TF-IDF"""
        # Création du corpus
        corpus = [" ".join(self.tokens)]
        if additional_docs:
            corpus.extend(additional_docs)
        
        # Vectorisation
        self.vectorizer = TfidfVectorizer(
            token_pattern=r'(?u)\b\w+\b',
            min_df=1,
            max_df=1.0,
            ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        return self
    
    def get_top_keywords(self, n=20):
        """Retourne les n mots-clés les plus importants"""
        feature_names = self.vectorizer.get_feature_names_out()
        weights = self.tfidf_matrix[0].toarray()[0]
        keywords = [(word, weight) for word, weight in zip(feature_names, weights) if weight > 0]
        keywords.sort(key=lambda x: x[1], reverse=True)
        return keywords[:n]
    
    def get_statistics(self):
        """Retourne les statistiques du pipeline"""
        return {
            'caracteres_bruts': len(self.raw_text),
            'caracteres_nettoyes': len(self.cleaned_text),
            'tokens_originaux': len(self.raw_text.split()) if self.raw_text else 0,
            'tokens_filtres': len(self.tokens),
            'vocabulaire_unique': len(set(self.tokens)),
            'mots_cles_top': self.get_top_keywords(5)
        }
    
    def run(self, additional_docs=None, min_word_length=2):
        """Exécute le pipeline complet"""
        print("="*60)
        print("PIPELINE NLP ARABE - DÉMARRAGE")
        print("="*60)
        
        print("\n[1/4] Chargement des données...")
        self.load_data()
        
        print("\n[2/4] Prétraitement...")
        self.preprocess(min_word_length)
        print(f"      {len(self.tokens)} tokens conservés")
        
        print("\n[3/4] Vectorisation TF-IDF...")
        self.vectorize(additional_docs)
        
        print("\n[4/4] Analyse terminée!")
        return self


In [47]:

#  8: Fonction utilitaire pour afficher les résultats
def display_results(pipeline):
    """Affiche les résultats du pipeline"""
    print("\n" + "="*60)
    print("RÉSULTATS DU PIPELINE")
    print("="*60)
    
    # Aperçu du texte
    print("\n--- APERÇU DU TEXTE NETTOYÉ ---")
    print(pipeline.cleaned_text[:300] + "..." if len(pipeline.cleaned_text) > 300 else pipeline.cleaned_text)
    
    # Tokens
    print("\n--- PREMIERS TOKENS (15) ---")
    for i, token in enumerate(pipeline.tokens[:15], 1):
        print(f"  {i}. {token}")
    
    # Mots-clés
    print("\n--- TOP 15 MOTS-CLÉS (TF-IDF) ---")
    keywords = pipeline.get_top_keywords(15)
    for i, (word, score) in enumerate(keywords, 1):
        print(f"  {i}. {word}: {score:.4f}")
    
    # Statistiques
    stats = pipeline.get_statistics()
    print("\n--- STATISTIQUES ---")
    print(f"  Caractères bruts: {stats['caracteres_bruts']}")
    print(f"  Caractères nettoyés: {stats['caracteres_nettoyes']}")
    print(f"  Tokens filtrés: {stats['tokens_filtres']}")
    print(f"  Vocabulaire unique: {stats['vocabulaire_unique']}")


In [48]:

#  9: Exemple d'utilisation avec votre PDF
# Remplacer 'text.pdf' par le chemin de votre fichier
pdf_file = "text.pdf"

# Création et exécution du pipeline
pipeline = ArabicNLPPipeline(pdf_path=pdf_file)

# Documents supplémentaires pour améliorer TF-IDF
additional_docs = [
    "الذكاء الاصطناعي تعلم الآلة معالجة البيانات",
    "تحليل البيانات الضخمة استخراج المعلومات",
    "تطوير أنظمة ذكية للتنبؤ واتخاذ القرارات"
]

# Exécution
pipeline.run(additional_docs=additional_docs, min_word_length=2)

# Affichage des résultats
display_results(pipeline)


PIPELINE NLP ARABE - DÉMARRAGE

[1/4] Chargement des données...
Extraction réussie: 1 pages, 1165 caractères

[2/4] Prétraitement...
      36 tokens conservés

[3/4] Vectorisation TF-IDF...

[4/4] Analyse terminée!

RÉSULTATS DU PIPELINE

--- APERÇU DU TEXTE NETTOYÉ ---
ُ ر ت ا ذ ء ا ط ن أھم ت ا و و ا د ، ث د ل ا ت ا وا راج ا ط ذ رارات د . دم ھذا ا ل وارز ت م ا ا و ت و طو ر أ ظ درة ا م وا ؤ وا م ن ا ر . ا وات ا رة، أ ا ذ ء ا ط زءاً أ ً ن ا و ، إذ م وظ ت ددة ل ا طب، وا م، وا ، وا رة ا رو . ل ا ل، د أ ظ ا ص ا ط ا ذ ا ط ء ا ف ا راض د أ ر، ُ دم ت ا و ت ل و ن ر ا دم. أن...

--- PREMIERS TOKENS (15) ---
  1. أھم
  2. راج
  3. رارات
  4. دم
  5. ھذا
  6. وارز
  7. طو
  8. درة
  9. وات
  10. رة،
  11. زءاً
  12. وظ
  13. ددة
  14. طب،
  15. م،

--- TOP 15 MOTS-CLÉS (TF-IDF) ---
  1. دم: 0.3180
  2. دام: 0.2120
  3. راج: 0.2120
  4. رارات: 0.2120
  5. رة: 0.2120
  6. ل: 0.2120
  7. ھذا: 0.2120
  8. أدوات: 0.1060
  9. أدوات راج: 0.1060
  10. أھم: 0.1060
  11. أھم راج: 0.1060
  12. داء: 0.1060
  

In [50]:

#  10: Export des résultats (optionnel)
def export_keywords_to_csv(pipeline, filename="keywords_arabic.csv"):
    """Exporte les mots-clés vers un fichier CSV"""
    keywords = pipeline.get_top_keywords(50)
    df = pd.DataFrame(keywords, columns=["Mot", "Score_TFIDF"])
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n✅ Mots-clés exportés vers {filename}")